In [ ]:
!pip install -q gradio groq

In [ ]:
import gradio as gr
import datetime
import json
import os
from groq import Groq
from google.colab import userdata

DB_FILE = "groq_calendar_db.json"

def load_calendar():
    if os.path.exists(DB_FILE):
        with open(DB_FILE, "r") as f:
            return json.load(f)
    return []

def save_calendar(data):
    with open(DB_FILE, "w") as f:
        json.dump(data, f, indent=4)

def web_agent_engine(user_query, selected_date):
    if not user_query.strip():
        return "Please type a scheduling instruction.", get_table_view()

    if selected_date:
        target_date = str(selected_date).split('T')[0]
    else:
        target_date = str(datetime.date.today())

    try:
        api_key = userdata.get('GROQPROJ_API_KEY')
        client = Groq(api_key=api_key)

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a schedule parser. Extract information from the request and return ONLY a valid JSON object with keys: 'title', 'start_time' (HH:MM), and 'end_time' (HH:MM). Default to '14:00' and '15:30' if times are missing. Do not output markdown, preambles, or formatting backticks. Just pure JSON."},
                {"role": "user", "content": f"Extract details from: '{user_query}'"}
            ],
            response_format={"type": "json_object"}
        )

        parsed_data = json.loads(response.choices[0].message.content)
        title = parsed_data.get("title", "Academic Session")
        start_time = parsed_data.get("start_time", "14:00")
        end_time = parsed_data.get("end_time", "15:30")

    except Exception as e:
        return f"🚨 Key Connection Error (Check your Colab 🔑 icon panel): {str(e)}", get_table_view()

    events = load_calendar()
    for e in events:
        if e['date'] == target_date:
            if not (end_time <= e['start_time'] or start_time >= e['end_time']):
                return f"🚨 CONFLICT DETECTED: Unable to schedule '{title}'. It overlaps with an existing entry on {target_date}: '{e['title']}' ({e['start_time']}-{e['end_time']}).", get_table_view()

    new_event = {
        "title": title,
        "date": target_date,
        "start_time": start_time,
        "end_time": end_time,
        "type": "Class"
    }
    events.append(new_event)
    save_calendar(events)
    return f"✅ Groq API Success! '{title}' added for {target_date} at {start_time}-{end_time}.", get_table_view()

def get_table_view():
    events = load_calendar()
    if not events:
        return [["No Events Scheduled", "-", "-", "-"]]
    sorted_events = sorted(events, key=lambda x: (x['date'], x['start_time']))
    return [[e['title'], e['date'], e['start_time'], e['end_time']] for e in sorted_events]

def clear_db():
    if os.path.exists(DB_FILE):
        os.remove(DB_FILE)
    return "Database Cleared!", [["No Events Scheduled", "-", "-", "-"]]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📅 Smart Timetable Assistant AI Website")
    gr.Markdown("Track A Execution Platform — Secure Groq API Integration Cloud Block")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 💬 Conversational Input Panel")
            event_date = gr.DateTime(label="Select Schedule Date", type="string")
            user_input = gr.Textbox(
                label="Instruct the Agent",
                placeholder="e.g., Schedule an AI Agent Lab session from 14:00 to 15:30"
            )
            submit_btn = gr.Button("Submit to Agent", variant="primary")
            output_msg = gr.Textbox(label="Agent Response Status", interactive=False)
            clear_btn = gr.Button("Reset Timetable Data", variant="stop")

        with gr.Column(scale=2):
            gr.Markdown("### 📋 Live Updated Timetable Dashboard")
            calendar_table = gr.Dataframe(
                headers=["Event Title", "Target Date", "Start Time", "End Time"],
                datatype=["str", "str", "str", "str"],
                value=get_table_view(),
                interactive=False
            )

    submit_btn.click(
        fn=web_agent_engine,
        inputs=[user_input, event_date],
        outputs=[output_msg, calendar_table]
    )
    clear_btn.click(fn=clear_db, inputs=None, outputs=[output_msg, calendar_table])

demo.launch(inline=True, share=False)

/tmp/ipykernel_666/4018805309.py:79: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>